In [1]:
from pycocotools.coco import COCO
import shutil
import os

coco_dir = r"C:\Users\achyu\COCO"
val_dir = os.path.join(coco_dir, "val2017")

# Load COCO annotations
coco = COCO(os.path.join(coco_dir, "annotations/instances_val2017.json"))

# Define relevant construction categories
construction_categories = ["person", "truck", "ladder", "street sign", "helmet", "crane", "excavator"]
category_ids = coco.getCatIds(catNms=construction_categories)

# Get image IDs for construction-related objects
image_ids = coco.getImgIds(catIds=category_ids)
images = coco.loadImgs(image_ids)

# Ensure output directory exists
construction_images_dir = os.path.join(coco_dir, "construction_images")
os.makedirs(construction_images_dir, exist_ok=True)

# Copy images containing construction-related objects to the new folder
for img in images:
    src_path = os.path.join(val_dir, img["file_name"])
    dest_path = os.path.join(construction_images_dir, img["file_name"])
    shutil.copy(src_path, dest_path)

print(f"Filtered and saved {len(images)} construction-related images.")

loading annotations into memory...
Done (t=1.04s)
creating index...
index created!
Filtered and saved 159 construction-related images.


In [2]:
import os
import shutil
import numpy as np
from pycocotools.coco import COCO

# Define COCO directories
coco_dir = r"C:\Users\achyu\COCO"
dataset_dirs = ["train2017", "val2017"]  # Removed "test2017" since COCO has no test annotations
construction_dir = os.path.join(coco_dir, "construction")

# Ensure subdirectories exist within "construction"
for dataset in dataset_dirs:
    os.makedirs(os.path.join(construction_dir, dataset), exist_ok=True)

# Define relevant construction categories
construction_categories = ["person", "truck", "ladder", "street sign", "helmet", "crane", "excavator"]

# Process each dataset
for dataset in dataset_dirs:
    dataset_path = os.path.join(coco_dir, dataset)
    annotation_file = os.path.join(coco_dir, f"annotations/instances_{dataset}.json")

    # Load COCO annotations
    coco = COCO(annotation_file)
    category_ids = np.array(coco.getCatIds(catNms=construction_categories))

    # Get image IDs for construction-related objects
    image_ids = np.array(coco.getImgIds(catIds=category_ids))
    images = coco.loadImgs(image_ids.tolist())  # Convert NumPy array back to list for COCO API

    # Use vectorized operations to filter and copy images
    [
        shutil.copy(
            os.path.join(dataset_path, img["file_name"]),
            os.path.join(construction_dir, dataset, img["file_name"])  # Save to respective folder
        )
        for img in images if os.path.exists(os.path.join(dataset_path, img["file_name"]))  # Ensure file exists
    ]

    print(f"✅ Filtered {len(images)} construction images from {dataset}.")

print("🎯 All construction-related images saved in:", construction_dir)

loading annotations into memory...
Done (t=27.07s)
creating index...
index created!
✅ Filtered 3992 construction images from train2017.
loading annotations into memory...
Done (t=1.03s)
creating index...
index created!
✅ Filtered 159 construction images from val2017.
🎯 All construction-related images saved in: C:\Users\achyu\COCO\construction


In [3]:
import os
import torch
import time
from ultralytics import YOLO
import random
from IPython.display import display

# Ensure GPU selection
device = "cuda" if torch.cuda.is_available() else "cpu"
display(f"Using device: {device}")
torch.cuda.set_device(0)  # Explicitly set the GPU device

# Set the local path to the construction dataset (inside COCO)
construction_dir = r"C:\Users\achyu\COCO\construction\val2017"  # Adjust if needed

# List all image files in the construction validation directory
image_files = [f for f in os.listdir(construction_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

# Select 5 random images
num_images = min(5, len(image_files))  # Ensure we don’t select more than available files
selected_images = random.sample(image_files, num_images) if image_files else []

if not selected_images:
    print("No image files found in the construction dataset.")
else:
    print(f"Using images: {selected_images}")

# Load YOLO Model
try:
    model = YOLO("yolo11n.pt").to(device)
    display("Model loaded successfully.")
except Exception as e:
    display(f"Error loading model: {e}")

# Running predictions on selected images
for image_name in selected_images:
    image_path = os.path.join(construction_dir, image_name)
    start_time = time.time()
    
    try:
        results = model.predict(source=image_path, save=True, imgsz=640, conf=0.25)
        results[0].show()
    except Exception as e:
        print(f"An error occurred during prediction on {image_name}: {e}")

'Using device: cuda'

Using images: ['000000377946.jpg', '000000192670.jpg', '000000242411.jpg', '000000102805.jpg', '000000313588.jpg']


'Model loaded successfully.'


image 1/1 C:\Users\achyu\COCO\construction\val2017\000000377946.jpg: 448x640 8 persons, 2 cars, 2 buss, 1 truck, 4 traffic lights, 152.5ms
Speed: 5.4ms preprocess, 152.5ms inference, 175.1ms postprocess per image at shape (1, 3, 448, 640)
Results saved to runs\detect\predict10

image 1/1 C:\Users\achyu\COCO\construction\val2017\000000192670.jpg: 448x640 6 persons, 5 cars, 1 baseball glove, 27.4ms
Speed: 4.5ms preprocess, 27.4ms inference, 4.3ms postprocess per image at shape (1, 3, 448, 640)
Results saved to runs\detect\predict10

image 1/1 C:\Users\achyu\COCO\construction\val2017\000000242411.jpg: 640x448 3 persons, 3 cars, 1 truck, 1 clock, 188.1ms
Speed: 2.0ms preprocess, 188.1ms inference, 4.0ms postprocess per image at shape (1, 3, 640, 448)
Results saved to runs\detect\predict10

image 1/1 C:\Users\achyu\COCO\construction\val2017\000000102805.jpg: 448x640 1 person, 3 cars, 1 truck, 29.0ms
Speed: 2.6ms preprocess, 29.0ms inference, 6.1ms postprocess per image at shape (1, 3, 448,

Above this will create a folder with construction specific images saved in its own folder

In [5]:
import os
import torch
import time
import cv2
import numpy as np
from ultralytics import YOLO
from IPython.display import display

# Ensure GPU selection
device = "cuda" if torch.cuda.is_available() else "cpu"
display(f"Using device: {device}")
torch.cuda.set_device(0)  # Explicitly set the GPU device

# Set the local path to the construction dataset (inside COCO)
construction_dir = r"C:\Users\achyu\COCO\construction\val2017"

# List all image files in the construction validation directory
image_files = [f for f in os.listdir(construction_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]

# Select 5 random images
num_images = min(5, len(image_files))
selected_images = [os.path.join(construction_dir, img) for img in random.sample(image_files, num_images)] if image_files else []

if not selected_images:
    print("No image files found in the construction dataset.")
else:
    print(f"Using images: {selected_images}")

# Load YOLO Model
try:
    model = YOLO("yolo11n.pt").to(device)
    display("Model loaded successfully.")
except Exception as e:
    display(f"Error loading model: {e}")

# Convert images to PyTorch tensors for batch inference
image_tensors = []
for image_path in selected_images:
    img = cv2.imread(image_path)
    img = cv2.resize(img, (640, 640))  # Resize for YOLO input
    img = torch.from_numpy(img).float().permute(2, 0, 1) / 255.0  # Normalize and convert to tensor
    image_tensors.append(img)

# Stack images into a batch tensor
batch_images = torch.stack(image_tensors).to(device)

# Run predictions on the batch
start_time = time.time()
try:
    results = model.predict(source=batch_images, save=True, imgsz=640, conf=0.25)
    for res in results:
        res.show()
except Exception as e:
    print(f"An error occurred during batch prediction: {e}")

print(f"Inference completed in {time.time() - start_time:.2f} seconds.")


'Using device: cuda'

Using images: ['C:\\Users\\achyu\\COCO\\construction\\val2017\\000000130699.jpg', 'C:\\Users\\achyu\\COCO\\construction\\val2017\\000000158548.jpg', 'C:\\Users\\achyu\\COCO\\construction\\val2017\\000000361551.jpg', 'C:\\Users\\achyu\\COCO\\construction\\val2017\\000000203294.jpg', 'C:\\Users\\achyu\\COCO\\construction\\val2017\\000000517069.jpg']


'Model loaded successfully.'


0: 640x640 7 persons, 1 truck, 1 frisbee, 44.1ms
1: 640x640 2 persons, 1 car, 1 dog, 44.1ms
2: 640x640 4 persons, 1 airplane, 2 trucks, 4 backpacks, 1 handbag, 1 suitcase, 44.1ms
3: 640x640 2 persons, 1 bus, 1 truck, 1 handbag, 44.1ms
4: 640x640 2 persons, 1 car, 1 bench, 1 chair, 44.1ms
Speed: 0.2ms preprocess, 44.1ms inference, 4.1ms postprocess per image at shape (1, 3, 640, 640)
Results saved to runs\detect\predict11
Inference completed in 1.48 seconds.


In [7]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("snehilsanyal/construction-site-safety-image-dataset-roboflow")

print("Path to dataset files:", path)

100%|██████████| 206M/206M [00:05<00:00, 36.2MB/s] 

Extracting files...


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\achyu\\.cache\\kagglehub\\datasets\\snehilsanyal\\construction-site-safety-image-dataset-roboflow\\versions\\3\\css-data\\train\\images\\1288788-une-employee-aide-des-voyageurs-en-provenance-de-chine-le-26-janvier-2020-a-l-aeroport-de-roissy_jpg.rf.35ed51188a266a262a24924aa32463a0.jpg'